# FR/SE trade-off — clean (isolated-only) vs noisy (wide-ε) 400-track events

**Todo:** *Demonstrate the segment efficiency vs false-rate trade-off is caused by ε
(competing-candidate segments).* We solve two 400-track events with **both** solvers
(classical exact $`A^{-1}\mathbf b`$ and the 1BQF) and compare their activation spectra
and segment metrics.

- **Event A — clean / isolated-only:** σ_scatt=0, σ_res=0, ε=1e-6 (numerical). True
  segments are exactly collinear → couple into 4-chains; cross-track false segments
  (θ≈mrad ≫ ε) stay **isolated**. So the *only* false segments are isolated.
- **Event B — noisy / fixed-ε scan:** σ_scatt=1e-4, σ_res=0, ε=2 mrad (event
  `ev_591b8b9b4b66`). Scattering + the wide acceptance produce **coupled** false
  clusters (cross-track bridges/hubs).

**Hypothesis:** when the only false segments are isolated, both solvers separate
true/false cleanly (classical keeps isolated false below τ; the 1BQF *erases* them at
the notch). Noise + a wider ε manufacture coupled false → the activation spectra
overlap and the false rate rises — the ε-driven SE↔FR trade-off.

In [1]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/_shared")
sys.path.insert(0, "/data/bfys/gscriven/LHCb_VeLo_Toy_Model/src")
import os; os.environ.setdefault("QTRK_STORE", "/data/bfys/gscriven/qtrk_store")
from pathlib import Path
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from collections import defaultdict
import matplotlib.pyplot as plt
import qtrk_pipeline as qp
from helpers import solve_quantum_statevector

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
OUT = Path("/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/FR_SE_tradeoff/outputs")
OUT.mkdir(parents=True, exist_ok=True)
G, D = 3.0, 1.0; SDIAG = G + D; TAU = qp.threshold_for(G, D)
GREEN, RED = "#1b7837", "#d6604d"
print("threshold τ =", TAU, "  notch λ = γ+δ =", SDIAG)

def cluster_struct(A):
    n = A.shape[0]
    C = (SDIAG*sp.identity(n, format="csr") - A.tocsr()); C.setdiag(0); C.eliminate_zeros()
    C = (abs(C) > 1e-9).astype(np.int8)
    deg = np.asarray(C.sum(1)).ravel()
    ncomp, lab = connected_components(C, directed=False)
    return deg, lab
def metrics(x, truth):
    a = x > TAU; nTA = int((truth & a).sum()); nFA = int((~truth & a).sum())
    return dict(eff=nTA/truth.sum(), far=nFA/max(a.sum(), 1), nFA=nFA, nTA=nTA, nact=int(a.sum()))

threshold τ = 0.35   notch λ = γ+δ = 4.0


In [2]:
# ---------- Event A: clean (generate, solve both) ----------
evA = qp.ensure_event(n_trk=400, rep=0, sigma_scatt=0.0, sigma_res=0.0, phi_max=0.2, hit_ineff=0.0)[0]
hamA = qp.build_hamiltonian(evA, epsilon=1e-6, gamma=G, delta=D); AA = hamA.A.tocsr()
truthA = np.asarray(qp.truth_from_event(evA), bool)
solCA, _ = qp.solve_classical(hamA)
degA, labA = cluster_struct(AA)
# quantum = real per-block 1BQF over the (block-diagonal) clean A; isolated -> 0 (notch). Cached.
cache = OUT / "eventA_xQ.npy"
if cache.exists():
    xQA = np.load(cache)
else:
    xQA = np.zeros(AA.shape[0]); comps = defaultdict(list)
    for i, l in enumerate(labA): comps[l].append(i)
    for mem in comps.values():
        if len(mem) == 1: continue
        m = np.array(mem); sub = AA[m][:, m].toarray()
        sol, _, _ = solve_quantum_statevector(sp.csr_matrix(sub), np.ones(m.size), device="CPU")
        xQA[m] = np.abs(np.asarray(sol[:m.size]))
    np.save(cache, xQA)
mask = solCA > TAU; xQA = xQA * (np.linalg.norm(solCA[mask]) / np.linalg.norm(xQA[mask]))   # signal-rescale
fcA = int((degA[~truthA] > 0).sum())
print(f"Event A: n_seg={AA.shape[0]}, n_true={int(truthA.sum())}, COUPLED-false={fcA} (all isolated if 0)")
print("  classical", metrics(solCA, truthA)); print("  quantum  ", metrics(xQA, truthA))

Event A: n_seg=640000, n_true=1600, COUPLED-false=0 (all isolated if 0)
  classical {'eff': np.float64(1.0), 'far': np.float64(0.0), 'nFA': 0, 'nTA': 1600, 'nact': 1600}
  quantum   {'eff': np.float64(0.75), 'far': np.float64(0.0), 'nFA': 0, 'nTA': 1200, 'nact': 1200}


In [3]:
# ---------- Event B: noisy (load existing event + stored real solves) ----------
sols = pd.read_csv(qp.manifest_dir() / "solutions.csv")
# select the FIXED-ε (set, ε=0.002) solves — the event is shared with the formula-ε study,
# so we must match the ε=2 mrad solves to the A we build at ε=0.002.
rowsB = sols[(sols.event_key == "ev_591b8b9b4b66") & (sols.gamma == 3.0) &
             (sols.eps_provenance == "set") & (sols.epsilon.round(6) == 0.002)]
kC = rowsB[rowsB.solver == "classical"].iloc[0].sol_key
kQ = rowsB[rowsB.solver == "quantum"].iloc[0].sol_key
print("Event B fixed-ε solves: classical", kC, "quantum", kQ,
      "| ε =", float(rowsB.iloc[0].epsilon))
evB = qp.load_event(qp.event_path("ev_591b8b9b4b66"))
hamB = qp.build_hamiltonian(evB, epsilon=0.002, gamma=G, delta=D); AB = hamB.A.tocsr()
truthB = np.asarray(qp.truth_from_event(evB), bool)
solCB = np.asarray(qp.load_solution(kC)["sol"], float)                 # real classical
xQB = qp.rescale_to_signal(np.asarray(qp.load_solution(kQ)["sol"], float), solCB, TAU)  # real GPU 1BQF, signal-rescaled
degB, labB = cluster_struct(AB)
fcB = int((degB[~truthB] > 0).sum())
print(f"Event B: n_seg={AB.shape[0]}, n_true={int(truthB.sum())}, COUPLED-false={fcB}")
print("  classical", metrics(solCB, truthB)); print("  quantum  ", metrics(xQB, truthB))

Event B fixed-ε solves: classical sol_94c79419fca4deba quantum sol_e6acbf5c5ff3395c | ε = 0.002


Event B: n_seg=640000, n_true=1600, COUPLED-false=1289
  classical {'eff': np.float64(1.0), 'far': np.float64(0.02020820575627679), 'nFA': 33, 'nTA': 1600, 'nact': 1633}
  quantum   {'eff': np.float64(0.74875), 'far': np.float64(0.017227235438884332), 'nFA': 21, 'nTA': 1198, 'nact': 1219}


## 1. The structural cause — isolated vs coupled false segments
$`A=(\gamma+\delta)I-C`$; a false segment is *isolated* (degree 0 in the compatibility
graph $`C`$) or *coupled*. Event A's tiny ε couples only the exactly-collinear true
segments, so **every false segment is isolated**; Event B's 2 mrad acceptance + scattering
couple cross-track segments into **bridge/hub clusters**.

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for axi, (deg, truth, nm, fc) in zip(ax, [(degA, truthA, "Event A (clean, ε=1e-6)", fcA),
                                           (degB, truthB, "Event B (noisy, ε=2 mrad)", fcB)]):
    dvf, dcf = np.unique(deg[~truth], return_counts=True)
    dvt, dct = np.unique(deg[truth], return_counts=True)
    axi.bar(dvf-0.18, dcf, 0.36, color=RED, label="false")
    axi.bar(dvt+0.18, dct, 0.36, color=GREEN, label="true")
    axi.set_yscale("log"); axi.set_xlabel("compatibility degree $k_i$"); axi.set_ylabel("# segments")
    axi.set_title(f"{nm}\ncoupled-false = {fc}", fontweight="bold", fontsize=10); axi.legend(fontsize=9)
fig.suptitle("Degree distribution: Event A has NO coupled false (all isolated); Event B does",
             fontweight="bold", y=1.0)
fig.tight_layout()
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"degree_structure.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show(); print("saved degree_structure")

saved degree_structure


## 2. Activation spectra — the heart of the trade-off
Histograms of the solver output $`x_i`$, true (green) vs false (red), for each event and
solver. The clean event separates true/false; the noisy event grows a **false tail into
the true band** — and for the 1BQF that tail overlaps the (notch-suppressed) true outer
plateau, so a single threshold can no longer separate them.

In [5]:
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
panels = [((0,0), solCA, truthA, "Event A — CLASSICAL"), ((0,1), xQA, truthA, "Event A — QUANTUM (1BQF)"),
          ((1,0), solCB, truthB, "Event B — CLASSICAL"), ((1,1), xQB, truthB, "Event B — QUANTUM (1BQF)")]
bins = np.linspace(0, 1.1, 90)
for (r, c), x, truth, nm in panels:
    a = ax[r, c]
    a.hist(x[~truth], bins=bins, color=RED, alpha=0.65, label=f"false ({(~truth).sum():,})")
    a.hist(x[truth], bins=bins, histtype="step", color=GREEN, lw=2.0, label=f"true ({truth.sum():,})")
    a.axvline(TAU, color="k", ls="--", lw=1.3, label=f"τ={TAU}")
    m = metrics(x, truth)
    mar = x[truth].min() - (np.percentile(x[~truth], 99.9) if (~truth).any() else 0)
    a.set_yscale("log"); a.set_ylim(0.5, None); a.set_xlabel("activation $x_i$"); a.set_ylabel("# segments")
    a.set_title(f"{nm}\neff={m['eff']*100:.0f}%  far={m['far']*100:.1f}%  (FP={m['nFA']})  margin={mar:+.2f}",
                fontsize=10, fontweight="bold")
    a.legend(fontsize=8, loc="upper right")
fig.suptitle("Activation spectra: clean → separated (false at 0/0.25); noisy → false tail into the true band",
             fontweight="bold", y=1.0)
fig.tight_layout()
for e, dp in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"activation_spectra.{e}", dpi=dp, bbox_inches="tight", facecolor="white")
plt.show(); print("saved activation_spectra")

saved activation_spectra


In [6]:
# ---------- metrics + separation-margin summary table ----------
rows = []
for nm, x, truth in [("A classical", solCA, truthA), ("A quantum", xQA, truthA),
                     ("B classical", solCB, truthB), ("B quantum", xQB, truthB)]:
    m = metrics(x, truth)
    mar = x[truth].min() - (np.percentile(x[~truth], 99.9) if (~truth).any() else 0)
    rows.append(dict(case=nm, eff=round(m["eff"], 3), far=round(m["far"], 4),
                     FP=m["nFA"], n_active=m["nact"],
                     min_true=round(x[truth].min(), 3), max_false=round(x[~truth].max(), 3),
                     margin=round(mar, 3)))
tab = pd.DataFrame(rows); tab.to_csv(OUT / "summary.csv", index=False)
print(tab.to_string(index=False))
print(f"\ncoupled-false segments:  Event A = {fcA}   Event B = {fcB}")

       case   eff    far  FP  n_active  min_true  max_false  margin
A classical 1.000 0.0000   0      1600     0.364      0.250   0.114
  A quantum 0.750 0.0000   0      1200     0.180      0.000   0.180
B classical 1.000 0.0202  33      1633     0.364      0.869   0.030
  B quantum 0.749 0.0172  21      1219     0.170      0.644  -0.127

coupled-false segments:  Event A = 0   Event B = 1289


## 3. The ε mechanism, and the conclusion
The only difference that turns Event A into Event B is **noise + a wider ε**:
the 2 mrad acceptance (needed so scattered *true* triplets still couple) simultaneously
admits *cross-track* pairs that share a hit, converting isolated false segments into
coupled bridge/hub clusters. Those coupled clusters are amplitude-degenerate with true
track structure (a false ≥3-chain has the same Hopfield activation as a real one), so
they cross τ — a **false positive** — in *both* solvers, and the 1BQF cannot remove them
because its notch only annihilates the *isolated* class.

**Result.**
- Event A (only isolated false): both solvers **far = 0**; the 1BQF erases the entire
  false bulk at the notch; clean positive separation margin.
- Event B (coupled false from noise + wide ε): **far > 0** for both; the quantum
  separation margin goes **negative** (true outer plateau ≈0.18 sits below the surviving
  false bridges) → true and false overlap.

So the segment-efficiency / false-rate trade-off is governed by ε: a narrow ε keeps the
false set isolated (far→0) but, under scattering, would also drop true segments
(efficiency loss); widening ε to recover those true segments is exactly what couples the
cross-track false segments and lifts the false rate. The 1BQF is *better* than classical
precisely in the regime where all false are isolated — and loses that advantage as soon
as ε + noise create competing (coupled) candidates.